In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
df = pd.read_csv('FeatureEngineering_assignment_data.csv')

X = df.drop('churn', axis=1)
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# ==========================================================
# Task 1: Custom Transformers
# ==========================================================

class TextToNumericCleaner(BaseEstimator, TransformerMixin):
    def __init__(self, column_name='total_charges'):
        self.column_name = column_name

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_out = X.copy()
        if self.column_name in X_out.columns:
            X_out[self.column_name] = pd.to_numeric(X_out[self.column_name], errors='coerce')
        return X_out

class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_out = X.copy()
        X_out['charge_per_month'] = X_out['total_charges'] / (X_out['tenure'] + 1)
        return X_out


In [ ]:
# ==========================================================
# Task 2: Advanced Preprocessing & Column Mapping
# ==========================================================

num_cols = ['tenure', 'monthly_charges', 'total_charges', 'charge_per_month', 'random_noise_1', 'random_noise_2']
cat_cols = ['contract_type', 'payment_method']

num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

In [ ]:
# ==========================================================
# Task 3 & 4: Full Pipeline & Joint Hyperparameter Tuning
# ==========================================================

# สร้าง Pipeline พื้นฐาน (กำหนด Feature Selection ตัวแรกเป็น Placeholder ไว้ก่อน)
full_pipeline = Pipeline(steps=[
    ('text_cleaner', TextToNumericCleaner(column_name='total_charges')),
    ('feature_engineer', FeatureEngineer()),
    ('preprocessor', preprocessor),
    ('feature_selection', SelectKBest(score_func=f_classif)), # Placeholder
    ('model', RandomForestClassifier(random_state=42))
])

# กำหนด Search Space สำหรับทดสอบทั้ง Filter Method และ Embedded Method ร่วมกัน
param_grid = [
    # Option A: Filter Method (SelectKBest)
    {
        'feature_selection': [SelectKBest(score_func=f_classif)],
        'feature_selection__k': [4, 6, 8],
        'model__n_estimators': [50, 100],
        'model__max_depth': [None, 5]
    },
    # Option B: Embedded Method (SelectFromModel ด้วย Lasso L1)
    {
        'feature_selection': [SelectFromModel(LogisticRegression(penalty='l1', solver='liblinear', random_state=42))],
        'feature_selection__threshold': ['mean', 'median'],
        'model__n_estimators': [50, 100],
        'model__max_depth': [None, 5]
    }
]

# ตั้งค่า 5-Fold Stratified CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1
)

print("กำลังทำการ Cross-Validation และค้นหา Feature Selection Strategy ที่ดีที่สุด...")
grid_search.fit(X_train, y_train)

In [ ]:
# ==========================================================
# Feature Inspection (ดึงรายชื่อคอลัมน์ที่ผ่านการเลือก)
# ==========================================================

best_pipeline = grid_search.best_estimator_

# 1. ดึงชื่อคอลัมน์ทั้งหมดหลังผ่าน Preprocessor
preprocessor_step = best_pipeline.named_steps['preprocessor']
feature_names_out = preprocessor_step.get_feature_names_out()

# 2. ดึง Boolean Mask จาก Feature Selection Step
selector_step = best_pipeline.named_steps['feature_selection']
selected_mask = selector_step.get_support()

# 3. สกัดเฉพาะชื่อ Feature ที่ถูกเลือก
selected_features = feature_names_out[selected_mask]

print("\n=========================================")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV F1-Score: {grid_search.best_score_:.4f}")
print("=========================================")
print(f"SELECTED FEATURES ({len(selected_features)} features):")
for feat in selected_features:
    print(f" - {feat}")
print("=========================================\n")

In [ ]:
# Evaluation บน Test Set
y_pred = best_pipeline.predict(X_test)
print("Test Set Classification Report:")
print(classification_report(y_test, y_pred))